# C4. Demo: Adnotare corpus și tipologii discursive

Acest notebook demonstrează workflow-ul complet C4:
1. Încărcăm corpusul YouTube curățat
2. Încărcăm promptul de adnotare de referință
3. Rulăm câteva exemple de adnotare cu LLM
4. Inspecăm outputul JSON
5. Construim tipologii discursive rule-based pe corpusul adnotat complet
6. Comparăm tipologia cu o explorare DBSCAN
7. Afișăm corpusul tipologizat

**Notă:** Adnotarea completă (420 comentarii) a fost rulată separat și salvată în `data/cleaned/corpus_youtube_sample_annotated.jsonl`. Demo-ul arată live adnotarea pe un subset mic, apoi lucrează pe corpusul complet adnotat.

## 0. Setup

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
MODEL = "gemini-2.5-flash"

print("Root:", ROOT)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model:", MODEL)

## 1. Încărcăm corpusul YouTube

In [ ]:
CORPUS_PATH = ROOT / "data" / "cleaned" / "corpus_youtube_sample.jsonl"

df = pd.read_json(CORPUS_PATH, lines=True)

print("Comentarii:", len(df))
print("Câmpuri:", list(df.columns))
df[["source_channel", "text"]].sample(3, random_state=1)

In [ ]:
# Distribuția canalelor sursă
df["source_channel"].value_counts().head(10)

## 2. Încărcăm promptul de adnotare de referință

In [ ]:
PROMPT_PATH = ROOT / "prompts" / "annotation_prompt.md"

ANNOTATION_PROMPT = PROMPT_PATH.read_text(encoding="utf-8")

print("Lungime prompt:", len(ANNOTATION_PROMPT), "caractere")
print()
# Afișăm primele 500 de caractere
print(ANNOTATION_PROMPT[:500])

## 3. Adnotare live — 5 exemple cu LLM

Trimitem 5 comentarii la model și inspecăm outputul JSON.

Schema de adnotare returnată de model:
```json
{
  "target": "",
  "stance": "",
  "tone": "",
  "institutional": 0,
  "legitimare": 0,
  "epistemic": 0,
  "geopolitic": 0,
  "mobilizare": 0,
  "justification": "",
  "confidence": 0.0
}
```

In [ ]:
def annotate(text, channel="", title="", client=gemini_client, model=MODEL):
    user_msg = f"""CANAL: {channel}
TITLU VIDEO: {title}
COMENTARIU: {text}"""
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=400,
        messages=[
            {"role": "system", "content": ANNOTATION_PROMPT},
            {"role": "user", "content": user_msg}
        ]
    )
    return response.choices[0].message.content

In [ ]:
# Alegem 5 comentarii reprezentative pentru demo
DEMO_SAMPLE = df.sample(5, random_state=42).copy()

demo_results = []
for _, row in DEMO_SAMPLE.iterrows():
    raw = annotate(row["text"], row.get("source_channel", ""), row.get("video_title", ""))
    print("=" * 70)
    print("CANAL:", row.get("source_channel", ""))
    print("COMENTARIU:", row["text"][:120])
    print("OUTPUT:")
    print(raw)
    demo_results.append({"id": row["id"], "text": row["text"], "raw": raw})

## 4. Parsăm și inspecăm outputul JSON

In [ ]:
def parse_json(raw):
    """Extrage JSON valid din răspunsul modelului."""
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"```$", "", raw.strip())
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # încearcă să extragă primul bloc JSON
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return None

parsed_demo = []
for r in demo_results:
    obj = parse_json(r["raw"])
    if obj:
        obj["id"] = r["id"]
        obj["text_preview"] = r["text"][:80]
        parsed_demo.append(obj)

pd.DataFrame(parsed_demo)[["text_preview", "target", "stance", "tone", "institutional", "epistemic", "confidence"]]

## 5. Tipologii discursive rule-based

Modelul nu decide bula discursivă. El produce variabile.
Scriptul construiește tipologia prin reguli pe combinația axelor.

| Tip | Logică |
|-----|--------|
| T1_suport_personalist | repr_personalist_strength >= 1 |
| T2_grievance_anti_sistem | inst_neg_strength >= 1, fără personalism |
| T3_opozitie_suveranista | stance=anti față de actori suveraniști |
| T4_conspiratie_externalism | epist_hidden_coordination >= 1 sau geo_anti >= 1 |
| T5_pro_democratic_european | inst_pos >= 1 sau geo_pro >= 1 |
| T6_afectiv_pozitional | restul (fără semnal discursiv dominant) |

Lucrăm pe corpusul complet adnotat:

In [ ]:
# Încărcăm corpusul complet adnotat
ANNOTATED_PATH = ROOT / "data" / "cleaned" / "corpus_youtube_sample_annotated.jsonl"

df_ann = pd.read_json(ANNOTATED_PATH, lines=True)

print("Comentarii adnotate:", len(df_ann))
print("Câmpuri:", list(df_ann.columns))

In [ ]:
# Distribuția tipurilor discursive
df_ann["discourse_type"].value_counts()

In [ ]:
# Distribuția subtipurilor discursive
df_ann["discourse_subtype"].value_counts().head(15)

In [ ]:
# Exemple de texte per tip discursiv
for tip in df_ann["discourse_type"].value_counts().index:
    print("\n" + "=" * 70)
    print(tip)
    print("=" * 70)
    sample = df_ann[df_ann["discourse_type"] == tip]["text"].dropna().head(3)
    for i, text in enumerate(sample, 1):
        print(f"{i}. {text[:100]}")

In [ ]:
# Profilul axelor per tip discursiv
AXE = [
    "inst_neg_strength", "inst_pos_strength",
    "epist_hidden_coordination_strength", "epist_evidence_verification_strength",
    "geo_anti_external_domination_strength", "geo_pro_external_anchoring_strength",
    "repr_personalist_strength", "repr_pluralist_strength",
    "call_to_action_strength"
]

# convertim la numeric
for col in AXE:
    df_ann[col] = pd.to_numeric(df_ann[col], errors="coerce").fillna(0)

df_ann.groupby("discourse_type")[AXE].mean().round(2)

## 6. Explorare DBSCAN — verificare exploratorie a tipologiei

DBSCAN este un algoritm de clustering nesupervizat.
Îl folosim ca verificare: dacă tipologia rule-based este coerentă,
clusterele DBSCAN ar trebui să se suprapună parțial cu tipurile T1–T5.

**Notă:** DBSCAN nu produce tipologii. Este doar o oglindă exploratorie.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Pregătim matricea de features
X = df_ann[AXE].values.astype(float)
X_scaled = StandardScaler().fit_transform(X)

# DBSCAN
db = DBSCAN(eps=1.5, min_samples=5).fit(X_scaled)
df_ann["dbscan_cluster"] = db.labels_

print("Clustere DBSCAN:", sorted(df_ann["dbscan_cluster"].unique()))
print(df_ann["dbscan_cluster"].value_counts())

In [ ]:
# Vizualizare PCA 2D — tipologie rule-based vs DBSCAN
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tipologie rule-based
types = df_ann["discourse_type"].unique()
colors_map = {t: plt.cm.tab10(i) for i, t in enumerate(types)}
for tip in types:
    mask = df_ann["discourse_type"] == tip
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=tip, alpha=0.6, s=20, color=colors_map[tip])
axes[0].set_title("Tipologie rule-based (PCA 2D)")
axes[0].legend(fontsize=6, loc="best")

# DBSCAN
clusters = df_ann["dbscan_cluster"].unique()
for c in sorted(clusters):
    mask = df_ann["dbscan_cluster"] == c
    label = f"Cluster {c}" if c != -1 else "Zgomot"
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=label, alpha=0.6, s=20)
axes[1].set_title("DBSCAN (PCA 2D)")
axes[1].legend(fontsize=7, loc="best")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation: tipologie vs DBSCAN
pd.crosstab(df_ann["discourse_type"], df_ann["dbscan_cluster"])

## 7. Corpusul tipologizat — afișare și export

Corpusul adnotat cu tipologii este gata pentru C5.
În C5, fiecare tip devine un agent care răspunde din perspectiva bulei sale.

In [ ]:
# Rezumat final al corpusului tipologizat
summary = df_ann.groupby("discourse_type").agg(
    n_texte=("id", "count"),
    confidence_medie=("confidence", "mean"),
    subtipuri=("discourse_subtype", "nunique")
).round(2)

print("Rezumat corpus tipologizat:")
summary

In [ ]:
# Afișăm câte un exemplu reprezentativ per tip (confidence ridicată)
keep_cols = ["id", "text", "source_channel", "discourse_type", "discourse_subtype",
             "target_refined", "stance_to_target", "confidence", "type_confidence"]

for tip in df_ann["discourse_type"].value_counts().index:
    ex = df_ann[(df_ann["discourse_type"] == tip) & (df_ann["type_confidence"] == "high")].head(1)
    if len(ex) == 0:
        ex = df_ann[df_ann["discourse_type"] == tip].head(1)
    print(f"\n{'='*70}")
    print(f"[{tip}]")
    row = ex.iloc[0]
    print(f"Subtype: {row['discourse_subtype']}")
    print(f"Target: {row['target_refined']} | Stance: {row['stance_to_target']}")
    print(f"Text: {row['text'][:200]}")

In [ ]:
# Corpusul tipologizat este deja salvat în:
print("Corpus adnotat:", ANNOTATED_PATH)
print("Gata pentru C5 — embeddings și vector store per bulă.")